In [38]:
from dotenv import load_dotenv
load_dotenv()


True

In [39]:
import os
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [40]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [41]:
len(embeddings.embed_query("Hello AI!"))

3072

In [42]:
from pinecone import Pinecone

In [43]:
pc = Pinecone()

In [44]:
from pinecone import ServerlessSpec


In [45]:
index_name = "traditional-rag-02"

In [46]:
#Creating the index
if not pc.has_index(index_name):
    pc.create_index(
    name = index_name,
    dimension = 3072,
    metric = "cosine",
    spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
    )


In [47]:
#Loading the Index
index = pc.Index(index_name)

In [48]:
from langchain_pinecone import PineconeVectorStore

In [49]:
vector_store = PineconeVectorStore(index = index, embedding = embeddings)

In [50]:
results = vector_store.similarity_search("what is a langchain?")

In [51]:
results

[Document(id='119f10e0-aeae-4ace-ab39-da1e017328ce', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='91d3c613-1652-4ca6-a8a4-5cd16a19abc6', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='88c9f91a-d163-4527-a658-a75d0f74f192', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.'),
 Document(id='4cd66369-34d6-4e5a-b446-2e1552aa89ff', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [52]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},#additional info
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)


In [53]:
documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [54]:
#universal indentification number
uuids = [str(uuid4()) for _ in range(len(documents))]

In [55]:
vector_store.add_documents(documents=documents, ids=uuids)

['9c10107d-40f7-46e9-8758-976bd7fd30f2',
 '4f5ceb1d-c3bb-4174-88b7-e81b51391e90',
 '07e849ac-5895-48fc-862c-b75f2b93ab11',
 '56ff0d2d-38e1-4d6b-a04a-091dfe4f6120',
 '9ffb2911-521e-4e31-9889-917534ac63fd',
 '51057322-a287-4321-9939-95b12ec582d7',
 '0f483c62-0f1e-4f50-a617-5684d959a19b',
 '507f3283-7528-4a5b-a12d-18dff1d1e0e7',
 '6da9b747-5ea3-4f27-b8e5-3a408e7468a1',
 '7b7f71f4-41ed-41a3-99bd-5d919e8f7786']

In [56]:
results = vector_store.similarity_search("what langchain provides to us?",k=1)

In [57]:
results

[Document(id='07e849ac-5895-48fc-862c-b75f2b93ab11', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!')]

In [58]:
results = vector_store.similarity_search("what langchain provides to us?",filter={"source": "tweet"})

In [59]:
results

[Document(id='07e849ac-5895-48fc-862c-b75f2b93ab11', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='119f10e0-aeae-4ace-ab39-da1e017328ce', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='91d3c613-1652-4ca6-a8a4-5cd16a19abc6', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='507f3283-7528-4a5b-a12d-18dff1d1e0e7', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [60]:
retriever=vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.7} #hyperparameter
)

In [61]:
retriever.invoke("langchain")

[Document(id='07e849ac-5895-48fc-862c-b75f2b93ab11', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='119f10e0-aeae-4ace-ab39-da1e017328ce', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='507f3283-7528-4a5b-a12d-18dff1d1e0e7', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='91d3c613-1652-4ca6-a8a4-5cd16a19abc6', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [72]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-3.6-flash')

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [63]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [64]:
prompt=PromptTemplate(
    template="""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:""",
    input_variables=['context', 'question']
)

In [65]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [73]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [74]:
rag_chain.invoke("what is langchain?")

"Based on the provided context, I don't know. The context mentions using LangChain for a project, but it does not explain what LangChain actually is."